In [1]:
from kernels import PhysKerBase, PhysKerGP
from priors import SquExpKernel
from sympy import *
import numpy as np

In [2]:
squexp = SquExpKernel(2) # Specifying the prior on u(x,t) ~ GP(0, SquExp)
L = sympify("- a*Derivative(u, x0, 2)") # Specifying the linear operator

b_ops = [sympify("Derivative(u, x0)"), sympify("Derivative(u, x0)")]
b_conds = [lambda x0,x1: x0==0, lambda x0,x1: x0==1]

In [3]:
phys_kernel = PhysKerBase(squexp, L, b_ops, b_conds)

In [4]:
X_u = np.column_stack((np.random.uniform(0,1,size=5), np.random.uniform(0,1,size=5)))
X_f = np.column_stack((np.random.uniform(0,1,size=5), np.random.uniform(0,1,size=5)))
X = (X_u, X_f)

In [5]:
K = phys_kernel.K_func(X)
K({"a":5, "w0":10, "w1":1, "s":1}, sigma=(0,0), diff_param="w0")

5 5
5 5


array([[-3.78576700e-270, -1.16934854e-002, -7.98124995e-003,
        -6.02143972e-003, -1.16147358e-002],
       [-1.16934854e-002, -5.91526093e-272, -1.40808687e-003,
        -5.89882011e-003, -9.30876910e-005],
       [-7.98124995e-003, -1.40808687e-003, -5.91526093e-272,
        -1.64902532e-003, -5.47484888e-004],
       [-6.02143972e-003, -5.89882011e-003, -1.64902532e-003,
        -9.24259520e-274, -3.88984612e-003],
       [-1.16147358e-002, -9.30876910e-005, -5.47484888e-004,
        -3.88984612e-003, -5.91526093e-272]])

### 1d HEAT EQUATION

In [2]:
def index_concat(u, f):
    u_indexed = np.column_stack((np.zeros(u.shape[0]), u))
    f_indexed = np.column_stack((np.ones(f.shape[0]), f))

    return np.vstack((u_indexed, f_indexed))

In [3]:
squexp = SquExpKernel(1) # Specifying the prior on u(x,t) ~ GP(0, SquExp)
L = sympify("-a*Derivative(u, x0, 2)") # Specifying the linear operator
phys_kernel = PhysKerBase(squexp, L)

In [4]:
phys_kernel_gp = PhysKerGP(phys_kernel, 
                           {"a":1, "w0":0.5, "s":0.5}, 
                           {"a":(1.0,1.0), "w0":(-100,100), "s":(0.1,10)})

In [5]:
true_f = np.vectorize(lambda x : (4*np.pi**2)*np.sin(2*np.pi*x))
true_u = np.vectorize(lambda x : np.sin(2*np.pi*x))

In [6]:
np.random.seed(3)

N_u = 2
N_f = 7

u_obs = np.array([0,1])

a,b = 0,1
n_f = N_f-1
indexes = np.arange(0,n_f+1)

f_obs = (a+b)/2 + ((b-a)/2)*np.cos(np.pi*indexes/n_f)

In [8]:
index_concat(u_obs, f_obs)

array([[0.       , 0.       ],
       [0.       , 1.       ],
       [1.       , 1.       ],
       [1.       , 0.9330127],
       [1.       , 0.75     ],
       [1.       , 0.5      ],
       [1.       , 0.25     ],
       [1.       , 0.0669873],
       [1.       , 0.       ]])

In [9]:
phys_kernel_gp.bounds

array([[   1. ,    1. ],
       [-100. ,  100. ],
       [   0.1,   10. ]])

In [10]:
phys_kernel_gp.theta

array([1. , 0.5, 0.5])

In [7]:
phys_kernel_gp(index_concat(u_obs, f_obs))

IndexError: index 1 is out of bounds for axis 1 with size 1

In [12]:
np.hstack(([[0., 1.]], [[1. ,       0.9330127, 0.75 ,     0.5       , 0.25      , 0.0669873, 0.       ]]))

array([[0.       , 1.       , 1.       , 0.9330127, 0.75     , 0.5      ,
        0.25     , 0.0669873, 0.       ]])

In [24]:
from sklearn.gaussian_process import GaussianProcessRegressor

gp = GaussianProcessRegressor(kernel=phys_kernel_gp,
                              random_state=2,
                              alpha=1e-6,
                              n_restarts_optimizer=2)

y = np.hstack((true_u(u_obs), true_f(f_obs)))
gp.fit(index_concat(u_obs, f_obs), y)

ValueError: all the input array dimensions except for the concatenation axis must match exactly, but along dimension 0, the array at index 0 has size 2 and the array at index 1 has size 7